![image.png](https://miro.medium.com/max/1400/1*5rairMYDqDpTmzytnqpYnQ.jpeg)
[Medium](https://medium.com/@naivetech/the-connection-between-spark-streaming-and-apache-kafka-with-java-9fcdc701a8ad)

## Spark Streaming + Kafka Integration Guide (Kafka broker version 0.10.0 or higher
[Link](https://spark.apache.org/docs/latest/streaming-kafka-0-10-integration.html)

:::: {.columns}

::: {.fragment .column width="50%"}
The Spark Streaming integration for Kafka 0.10 provides simple parallelism, 1:1 correspondence between Kafka partitions and Spark partitions, and access to offsets and metadata. However, because the newer integration uses the new [Kafka consumer API](https://kafka.apache.org/documentation.html#newconsumerapi) instead of the simple API, there are notable differences in usage.
::: 

::: {.fragment .column width="50%"}
![](https://i.imgflip.com/57d4er.jpg)
[NicsMeme](https://imgflip.com/i/57d4er)
:::
::::


## Structured Streaming + Kafka Integration Guide (Kafka broker version 0.10.0 or higher)
[Link](https://spark.apache.org/docs/latest/structured-streaming-kafka-integration.html)

Structured Streaming integration for Kafka 0.10 to read data from and write data to Kafka.

:::: {.columns}

::: {.fragment .column width="50%"}
**Linking**
For Scala/Java applications using SBT/Maven project definitions, link your application with the following artifact:

```properties
groupId = org.apache.spark
artifactId = spark-sql-kafka-0-10_2.13
version = 4.3.0
```

- Match the connector version and Scala binary version to the Spark runtime (the demo uses Spark 4.3.0 and Scala 2.13).
- For Python applications, add this above library and its dependencies when deploying application.
- For spark-shell, add this above library and its dependencies too when invoking spark-shell
::: 

::: {.fragment .column width="50%"}
**Example** 

```bash
cd quickstart/kafka-pyspark-structured-streaming-demo
docker compose up

docker exec -it broker /bin/bash
/opt/kafka/bin/kafka-console-producer.sh --bootstrap-server broker:9092 --topic tap

```
:::
::::

## Creating a Kafka Source

:::: {.columns}

::: {.fragment .column width="50%"}
**Streaming Queries**

Standard approach, connects Kafka topics (one or more) and run queries

```python
# Subscribe to 1 topic
df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "host1:port1,host2:port2") \
  .option("subscribe", "topic1") \
  .load()
df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
```
::: 

::: {.fragment .column width="50%"}
**Batch Queries**

If you have a use case that is better suited to batch processing, you can create a Dataset/DataFrame for a defined range of offsets.

```python
df = spark \
  .read \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "host1:port1,host2:port2") \
  .option("subscribe", "topic1") \
  .load()
df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
```
:::
::::


### Row format

| Column             | Type      |
|--------------------|-----------|
| key                | binary    |
| value              | binary    |
| topic              | string    |
| partition          | int       |
| offset             | long      |
| timestamp          | timestamp |
| timestampType      | int       |
| headers (optional) | array     |

## Kafka Required Options
| Option                  | value                                       | meaning                                                                                                                                      |
|-------------------------|---------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------|
| assign                  | json string {"topicA":[0,1],"topicB":[2,4]} | Specific TopicPartitions to consume. Only one of "assign", "subscribe" or "subscribePattern" options can be specified for Kafka source.      |
| subscribe               | A comma-separated list of topics            | The topic list to subscribe. Only one of "assign", "subscribe" or "subscribePattern" options can be specified for Kafka source.              |
| subscribePattern        | Java regex string                           | The pattern used to subscribe to topic(s). Only one of "assign, "subscribe" or "subscribePattern" options can be specified for Kafka source. |
| kafka.bootstrap.servers | A comma-separated list of host:port         | The Kafka "bootstrap.servers" configuration.                                                                                                 |

## Kafka Optional Options: where to start

- `startingOffsets`: `latest` (default for streaming), `earliest` (default for batch), or offsets per topic/partition in JSON.
- In offset JSON, `-2` means earliest and `-1` means latest. A batch **start** cannot use latest.
- `startingOffsetsByTimestamp`: choose the start timestamp for each topic/partition; it takes precedence over `startingOffsets`.

::: {.callout-note .fragment}
Starting options apply to a **new** streaming query. On restart, Spark resumes from the checkpoint; newly discovered partitions start at earliest.
:::

## Kafka Optional Options: a bounded batch

- `endingOffsets` controls where a **batch** query stops; its default is latest.
- Use `latest` or per-partition JSON; `-1` means latest. Earliest (`-2`) is not allowed as a batch end.
- `endingOffsetsByTimestamp` chooses the end timestamp per topic/partition and takes precedence over `endingOffsets`.

:::{.fragment}
Example: read the offsets currently retained for a topic:

```python
batch = (spark.read.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "tap")
    .option("startingOffsets", "earliest")
    .option("endingOffsets", "latest")
    .load())
```
:::

## Kafka Optional Options: throughput and recovery

- `maxOffsetsPerTrigger`: maximum offsets per streaming trigger, distributed across topic partitions. Default: no limit.
- `minPartitions`: desired minimum Spark read partitions; Spark may split large Kafka partitions. Default: Kafka topic-partition count.
- `failOnDataLoss`: fail if data may have been lost, for example because offsets were removed. Default: `true`.
- `kafkaConsumer.pollTimeoutMs`: executor Kafka polling timeout; falls back to `spark.network.timeout`.
- `fetchOffset.numRetries` and `fetchOffset.retryIntervalMs`: retries and delay when fetching offsets.

## Kafka Optional Options: groups and headers

- `groupIdPrefix`: prefix for Spark-generated consumer group IDs; default `spark-kafka-source`.
- `kafka.group.id`: explicitly sets a group ID. Use with care: queries sharing an ID can interfere with each other. When set, it overrides `groupIdPrefix`.
- `includeHeaders`: include Kafka record headers in the DataFrame; default `false`.

For the complete, version-specific option list and defaults, see the [Spark 4.3 Kafka integration guide](https://spark.apache.org/docs/4.3.0/streaming/structured-streaming-kafka-integration.html#reading-data-from-kafka).

## Kafka Configuration: options and groups

Pass Kafka client settings with the `kafka.` prefix:

```python
.option("kafka.bootstrap.servers", "broker:9092")
```

Some settings are managed by Spark rather than configured as usual. Spark generates a unique consumer group ID per query; `groupIdPrefix` changes its prefix. An explicit `kafka.group.id` is possible, but sharing one between queries can cause interference.

## Kafka Configuration: offsets and recovery

- Use Spark's `startingOffsets`, not Kafka's `auto.offset.reset`.
- Starting offsets apply to a **new** query; a restarted query resumes from its checkpoint.
- Spark manages source progress; the Kafka source does not commit offsets using `enable.auto.commit`.

::: {.callout-note}
Kafka retention may remove offsets before Spark reads them. `failOnDataLoss` helps detect possible data loss; a checkpoint cannot recover records that are no longer retained.
:::

## Kafka Configuration: keys and values

- The Kafka source reads keys and values as **byte arrays**. Spark manages the Kafka deserializers; convert the columns with DataFrame operations.
- For Kafka writes, provide keys and values as **strings or byte arrays**. Spark manages the Kafka serializers.
- Consumer `interceptor.classes` is not supported because interceptors can interfere with the expected byte-array records.

```python
decoded = df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)")
```

See [Kafka-specific configuration in Spark 4.3](https://spark.apache.org/docs/4.3.0/streaming/structured-streaming-kafka-integration.html#kafka-specific-configurations).

## Deploying

As with any Spark application, spark-submit is used to launch your application. Add the connector and its dependencies with --packages; its Spark and Scala versions must match the runtime:

```bash
./bin/spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0 path/to/application.py
```

For experimenting on spark-shell, use the matching connector and its dependencies:

```bash
./bin/spark-shell --packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.3.0
```

# Spark Structured Streaming, Kafka and Schema Registry

:::{.callout-note}
This example has been vibe coded, starting from a prompt, looking at the slide content and refined through multiple iterations.
:::

## Pizza orders: the pipeline

```{mermaid}
flowchart LR
    D["Datagen: pizza_orders"] --> C["Kafka Connect<br/>AvroConverter"]
    C -->|"Avro records"| K["Kafka topic<br/>2 brokers, 2 partitions"]
    C -->|"register schema"| R["Schema Registry"]
    K -->|"read stream"| S["Spark Structured Streaming<br/>2 workers"]
    S -->|"look up schema"| R
    S --> P["Parquet files<br/>pizza_data volume"]
    S --> Q["Checkpoint<br/>spark-data volume"]
```

The Kafka record contains a Confluent framing header and an Avro payload. Spark retrieves the schema, verifies the schema ID, and decodes the payload before writing Parquet.

## Step 1: start the infrastructure

From the course repository root, with Docker Compose, `curl`, internet access, and free local ports 18080, 18081, 18083 and 14040:

```bash
cd quickstart/pizza-kafka-spark
docker compose up -d --build --wait broker1 broker2 registry connect
docker compose ps
```

The first build downloads the Datagen plugin. Kafka has two brokers; broker 1 is also the only KRaft controller, so this is a **development setup, not an HA deployment**.

## Step 2: create the topic and start Datagen

Create a topic with two partitions and two replicas, then register the connector:

```bash
docker compose exec broker1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server broker1:9092 --create --if-not-exists \
  --topic pizza_orders --partitions 2 --replication-factor 2

curl --fail-with-body -X PUT -H 'Content-Type: application/json' \
  --data @connector.json http://localhost:18083/connectors/pizza-orders/config
```

`connector.json` selects the `pizza_orders` quickstart and uses `store_id` as the Kafka key, so different stores can be distributed across partitions. The Connect worker's Avro converter registers the value schema with Schema Registry.

## Step 3: verify the source and schema

```bash
curl -fsS http://localhost:18083/connectors/pizza-orders/status
curl -fsS http://localhost:18081/subjects/pizza_orders-value/versions/latest
docker compose exec broker1 /opt/kafka/bin/kafka-topics.sh \
  --bootstrap-server broker1:9092 --describe --topic pizza_orders
```

Wait for the connector **and its task** to report `RUNNING`, and for the schema request to return an ID and Avro schema. The subject may take a few seconds to appear after registration. If either check fails, inspect `docker compose logs connect registry` before continuing.

## Step 3 continued: see the Avro records

Use the Registry-aware console consumer to decode and display three orders:

```bash
docker compose exec registry kafka-avro-console-consumer \
  --bootstrap-server broker1:9092,broker2:9092 \
  --topic pizza_orders --from-beginning --max-messages 3 \
  --formatter-property schema.registry.url=http://registry:8081
```

The displayed JSON is the **decoded Avro record**, not the raw Kafka value. The consumer reads the schema ID in the Confluent header and retrieves the schema from the Registry.

## Step 4: stream Kafka to Parquet

```bash
docker compose --profile spark up -d spark
docker compose logs -f spark
```

The command starts a Spark standalone master, two workers with one core each, and a driver requesting two one-core executors. The application fetches the latest registered schema, checks each value's five-byte Confluent header, and uses Spark's `from_avro` to decode the payload. It writes Parquet in append mode with a persistent checkpoint.

This demo expects a **single schema ID**; a different ID or malformed record stops the query explicitly. Handling multiple schema versions requires a more complete schema-aware design.

## Step 5: inspect and restart

After Spark has processed a batch, query the Parquet output:

```bash
docker compose exec spark /opt/spark/bin/spark-sql -e \
  "SELECT kafka_partition, kafka_offset, store_id, store_order_id, status FROM parquet.\`/opt/spark/work-dir/pizza-demo/parquet\` LIMIT 5"

docker compose stop spark
docker compose --profile spark up -d spark
```

The `spark-data` volume retains the checkpoint; the separate `pizza_data` volume holds Parquet files and is mounted on the driver and both workers. On restart, Spark resumes checkpointed offsets; `startingOffsets=earliest` is for the initial query only. Check the logs for errors and new batches.

::: {.callout-warning}
If you ran the earlier single-volume demo, its Parquet files are still in `spark-data` but are hidden by the new `pizza_data` mount. Do not resume its old checkpoint against an empty output volume: migrate the existing Parquet output (including `_spark_metadata`) or start a fresh demo after preserving any data you need.
:::

## Observe parallelism

- Open the [Spark master UI](http://localhost:18080): check that **two workers** are alive and each has an executor for the application.
- Open the [Spark application UI](http://localhost:14040): inspect **Executors** and a streaming batch's **Stages/Tasks**.
- With this default Kafka source configuration, two Kafka partitions map to two Spark input partitions; when both have records, the batch has two input tasks.

::: {.callout-note}
The 1:1 mapping is **Kafka partitions to Spark input partitions**, not a permanent broker-to-executor assignment. The scheduler chooses where tasks run, and a batch may have fewer active tasks if a Kafka partition has no new records. Confirm actual task placement in the UI.
:::

## Step 6: discuss and stop

- Kafka retains the events; the Registry stores the schema; Spark reads and transforms; Parquet stores the results.
- The checkpoint tracks progress; `pizza_data` is a shared Docker volume containing Parquet output. This single-host volume is **not distributed storage** or a backup.
- The next lesson asks how shared storage and a table format such as Iceberg can manage data files for analytical use.

```bash
docker compose --profile spark down
```

Named volumes are retained for restart. To delete this demo's data and start fresh, use `docker compose --profile spark down -v` **only when you want to delete those volumes**.

# Biblio

* https://www.slideshare.net/databricks/writing-continuous-applications-with-structured-streaming-python-apis-in-apache-spark
* https://kontext.tech/column/streaming-analytics/475/spark-structured-streaming-read-from-and-write-into-kafka-topics
* https://medium.com/knoldus/kafka-and-spark-streams-the-happily-ever-after-84b6f4b6b509
